In [11]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np

In [12]:
def filter_2023(df):
    df['datetime'] = pd.to_datetime(df['datetime'])
    return df[df['datetime'].dt.year == 2023]

In [13]:
def process_dataset(file_path):
    df = pd.read_csv(file_path)
    if 'datetime' not in df.columns:
        return None
    
    df = filter_2023(df)
    
    if df.empty:
        return None
    
    df = df.copy()  # Explicitly create a copy of the DataFrame

    event_name = os.path.splitext(os.path.basename(file_path))[0]
    df.loc[:, 'hour'] = df['datetime'].dt.hour
    df.loc[:, 'day_of_week'] = df['datetime'].dt.dayofweek
    df.loc[:, 'day_of_month'] = df['datetime'].dt.day
    df.loc[:, 'month'] = df['datetime'].dt.month
    df.loc[:, 'month_name'] = df['datetime'].dt.month_name()
    
    avg_hour = df['hour'].mean()
    avg_day_of_week = df['day_of_week'].mean()
    avg_day_of_month = df['day_of_month'].mean()
    
    day_names = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    avg_day_name = day_names[int(round(avg_day_of_week))]

    monthly_counts = df['month_name'].value_counts().sort_index()
    
    return {
        'event_name': event_name,
        'avg_hour_of_day': avg_hour,
        'avg_day_of_week': avg_day_name,
        'avg_day_of_month': avg_day_of_month,
        'count': monthly_counts.to_dict()  # Store monthly counts as a dictionary
    }

In [14]:
def process_all_datasets(directory):
    results = []
    for filename in os.listdir(directory):
        if filename.endswith('.csv'):
            file_path = os.path.join(directory, filename)
            result = process_dataset(file_path)
            if result:
                results.append(result)
    
    results_df = pd.DataFrame(results)
    
        # Expand the 'count' column into separate columns for each month
    counts_df = results_df['count'].apply(pd.Series).fillna(0)
    results_df = results_df.drop(columns=['count']).join(counts_df)
    
    output_path = os.path.join('E:\Economic_Data\Output data\output', 'average_event_times.csv')
    results_df.to_csv(output_path, index=False)
    return results_df

In [15]:
def plot_event_counts(df, output_directory, prefix=''):
    if not os.path.exists(output_directory):
        os.makedirs(output_directory)
    
    df = df.copy()  # Explicitly create a copy of the DataFrame
    df.loc[:, 'hour'] = df['datetime'].dt.hour
    df.loc[:, 'day_of_week'] = df['datetime'].dt.dayofweek
    df.loc[:, 'day_of_month'] = df['datetime'].dt.day
    df.loc[:, 'month'] = df['datetime'].dt.month
    df.loc[:, 'month_name'] = df['datetime'].dt.month_name()
    
    day_names = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    df.loc[:, 'day_name'] = df['datetime'].dt.day_name()

    def plot_and_save(data, title, xlabel, ylabel, filename, labels=None, xticks=None):
        plt.figure(figsize=(12, 8))
        data.plot(kind='bar', color='skyblue')
        plt.title(title)
        plt.xlabel(xlabel)
        plt.ylabel(ylabel)
        if xticks is not None:
            plt.xticks(range(len(xticks)), xticks)
        if labels:
            for i, count in enumerate(data):
                if np.isfinite(count):
                    plt.text(i, count + 0.1, str(count), ha='center')
        plt.savefig(os.path.join(output_directory, filename))
        plt.close()

    # Overall counts
    hour_counts = df['hour'].value_counts().sort_index()
    hour_counts = hour_counts.reindex(range(24), fill_value=0)  # Ensure all hours are included
    plot_and_save(hour_counts, f'{prefix} Count of Events by Hour of Day in 2023', 'Hour of Day', 'Count of Events', f'{prefix}_events_by_hour.png', labels=True, xticks=[f'{i:02}:00' for i in range(24)])

    day_counts = df['day_name'].value_counts().reindex(day_names, fill_value=0)
    plot_and_save(day_counts, f'{prefix} Count of Events by Day of Week in 2023', 'Day of Week', 'Count of Events', f'{prefix}_events_by_day_of_week.png', labels=True, xticks=day_names)

    for month in df['month'].unique():
        month_df = df[df['month'] == month]
        if not month_df.empty:
            days_in_month = month_df['datetime'].dt.days_in_month.iloc[0]
            day_of_month_counts = month_df['day_of_month'].value_counts().sort_index()
            day_of_month_counts = day_of_month_counts.reindex(range(1, days_in_month + 1), fill_value=0)  # Ensure all days are included
            month_name = month_df['month_name'].iloc[0]
            plot_and_save(day_of_month_counts, f'{prefix} Count of Events by Day of Month in 2023 - {month_name}', 'Day of Month', 'Count of Events', f'{prefix}_events_by_day_of_month_{month_name}.png', labels=True, xticks=range(1, days_in_month + 1))


In [16]:
# Directory containing all datasets
input_directory = 'E:\Economic_Data\Output data'
output_directory = 'E:\Economic_Data\Output data\output'

# Process all datasets to consider only events from 2023
results = []
for filename in os.listdir(input_directory):
    if filename.endswith('.csv'):
        file_path = os.path.join(input_directory, filename)
        df = pd.read_csv(file_path)
        if 'datetime' in df.columns:
            df_2023 = filter_2023(df)
            results.append(df_2023)

In [17]:
# Combine all filtered data into a single DataFrame
combined_df = pd.concat(results, ignore_index=True)

In [18]:
# Plot overall event counts
plot_event_counts(combined_df, output_directory)

In [19]:
# Plot event counts for each file separately
for filename in os.listdir(input_directory):
    if filename.endswith('.csv'):
        file_path = os.path.join(input_directory, filename)
        df = pd.read_csv(file_path)
        if 'datetime' in df.columns:
            df_2023 = filter_2023(df)
            if not df_2023.empty:
                event_name = os.path.splitext(filename)[0]
                plot_event_counts(df_2023, output_directory, prefix=event_name)

In [20]:
# Process all datasets and save results to CSV
results_df = process_all_datasets(input_directory)